# JAO Data Preprocessing

**Time Index**

* Convert all indices to UTC before merging or comparing the datasets.
* Resample all datasets to a common hourly UTC grid (mean of 15-min values in `allocation_constraint`, which has mixed 15-min and hourly steps), which also turns missing timestamps into missing values.

**Missing timestamps**

* `minmax_np` – no gaps, nothing to fill.
* Gaps of up to 3 hours (4 isolated single-hour gaps in `d2cf`) – linear interpolation.
* Longer gaps (all are whole-day gaps) – fill with values from the same hours one week earlier (D-7):
  * `refprog` and `d2cf`: 2023-10-30, 2024-06-25, 2024-10-28,
  * `allocation_constraint`: 2023-10-30, 2024-10-28.
* D-7 is used because JAO series (exchange programs, net positions, grid-model load) follow a weekly demand and market cycle, and most of the missing days are Mondays, whose previous day is a Sunday with a different profile (for 2023-10-30 and 2024-10-28 also a 25-hour DST day). For 2023-10-30 and 2024-10-28 the source day lies on the other side of the autumn time switch, so the copied hours are shifted by one hour relative to local time. Interpolation over 24 hours would flatten the daily profile, so it is limited to gaps of up to 3 hours (the same limit as in the ENTSO-E notebook).

In [1]:
import pandas as pd
import logging
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))
from src.config import DATA_DIR, DOWNLOAD_START_DATE, DOWNLOAD_END_DATE
from src.utils.data_quality import print_missing_values, print_index_summary

from src.utils.preprocessing import preprocess_jao
from src.utils.file_io import inspect_jao_files, load_jao_data, save_data

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
logging.basicConfig(level=logging.INFO)

In [3]:
data_jao = load_jao_data(DATA_DIR / "clean" / "jao" / date_range)

In [4]:
data_jao = preprocess_jao(data_jao)

## 1. Index Integrity – Confirmation

All four datasets now share a UTC index with a single 1-hour step, without duplicated timestamps and without Summer/Winter time repeats, so the gaps in the time index are gone.

In [5]:
for name, data in data_jao.items():
    print_index_summary(data=data, name=name)


--- [refprog] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 33238

--- [minmax_np] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 33262

--- [d2cf] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 33238

--- [allocation_constraint] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 33238


## 2. Missing Values – Confirmation

No missing values remain in any dataset: the single-hour gaps in `d2cf` were interpolated and the whole-day gaps were filled with the values from one week earlier.

In [6]:
for name, data in data_jao.items():
    print_missing_values(data=data, name=name)


--- [refprog] Missing values ---
No missing values.

--- [minmax_np] Missing values ---
No missing values.

--- [d2cf] Missing values ---
No missing values.

--- [allocation_constraint] Missing values ---
No missing values.


## 3. Conclusions

The preprocessing decisions from the data quality audit have been applied to all four JAO datasets, and the checks above confirm the result. The preprocessed datasets are saved to `data/preprocessed/jao/` and verified by reading the files back.

## 4. Saving Preprocessed Data

In [7]:
save_data(data_jao, DATA_DIR / "preprocessed" / "jao" / date_range)

INFO:src.utils.file_io:Saved preprocessed/jao/2022-12-01_2026-09-16/refprog.parquet
INFO:src.utils.file_io:Saved preprocessed/jao/2022-12-01_2026-09-16/minmax_np.parquet
INFO:src.utils.file_io:Saved preprocessed/jao/2022-12-01_2026-09-16/d2cf.parquet
INFO:src.utils.file_io:Saved preprocessed/jao/2022-12-01_2026-09-16/allocation_constraint.parquet


In [8]:
inspect_jao_files(DATA_DIR / "preprocessed" / "jao" / date_range)


--- JAO (4 file/s) ------------------------------
  [FILE] data/preprocessed/jao/2022-12-01_2026-09-16/refprog.parquet | shape: (33239, 65) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 21:00:00+00:00
  [FILE] data/preprocessed/jao/2022-12-01_2026-09-16/minmax_np.parquet | shape: (33263, 28) | idx: 2022-11-30 23:00:00+00:00-2026-09-16 21:00:00+00:00
  [FILE] data/preprocessed/jao/2022-12-01_2026-09-16/d2cf.parquet | shape: (33239, 51) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 21:00:00+00:00
  [FILE] data/preprocessed/jao/2022-12-01_2026-09-16/allocation_constraint.parquet | shape: (33239, 2) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 21:00:00+00:00
